In [1]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import netCDF4 as n
import glob
import xarray as xr
import seaborn as sns
import matplotlib.lines as mlines
import pandas as pd
import function_sampling_Thermal_forcing_files as fn
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.stats import linregress
from collections import Counter
sns.set_context("paper")
sns.set_style("whitegrid")
computed_name = 'ComputedScalarsHelene'
variable2 = 'dslc_anom'

In [2]:
 
mnt_pth = '/Volumes/CrucialX8/computing/data/antarctica/' #mount path
pth_imip6hack= mnt_pth + 'ismip6_hackathon/ComputedScalars/' #write folder
pth_ismip6 =mnt_pth + 'ismip6_hackathon/' #read folder
pth_helene =mnt_pth + 'ismip6_hackathon/ComputedScalarsHelene/'
 
expnames = ['expAE02', 'expAE03', 'expAE04', 'expAE05']
removeFileID = ['IMAU_UFEMISM1', 'IMAU_UFEMISM2', 'IMAU_UFEMISM3', 'IMAU_UFEMISM4',
               'DOE_MALI_4km', 'DOE_MALI_8km_Ant95', 'DOE_MALI_8km_AntMean'] # specify models to remove
 
metadata_file = 'Metadata.txt'
# Generate loop_info DataFrame

In [3]:
def get_model(string,exp):
    return string.split('/')[-1].split('AIS_')[-1].split('_'+str(exp))[0]

In [4]:
# avg and abs 2100
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    #get shelf area
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareafl/computed_iareafl_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    pth_tf = f'{pth_imip6hack}/{df.iloc[i].Experiment}/{variable2}/computed_{variable2}_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'

    #thermal foring 
    d3 = xr.open_dataset(pth_tf,decode_times=False )

    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )
    #
    ind = np.where((d.time.values >=2070) & (d.time.values <= 2100))
    
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =-d3[variable2].values[:tmin] #GT/m^3 yr

   
    yais =y[ind].mean()
  
    
    df['AIS'].iloc[i]=  yais
   

    for j in range(1,19):
        y =-1*d3[f'{variable2}_sector_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = y[ind].mean()
        
    for j in range(1,4):
        y =-1*d3[f'{variable2}_region_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = y[ind].mean()
        
        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
df.to_csv(f'tables/{variable2}_2100.csv', index=False)


/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_61411/3748087036.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]=  yais
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_61411/3748087036.py:59: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sector_'+str(j)].iloc[i] = y[ind].mean()
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_61411/3748087036.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-vie

In [6]:
# avg and abs 2100
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    #get shelf area
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareafl/computed_iareafl_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    pth_tf = f'{pth_imip6hack}/{df.iloc[i].Experiment}/{variable2}/computed_{variable2}_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'

    #thermal foring 
    d3 = xr.open_dataset(pth_tf,decode_times=False )

    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )
    #
    ind = np.where((d.time.values >=2170) & (d.time.values <= 2200))
    
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =-d3[variable2].values[:tmin] #GT/m^3 yr

   
    yais =y[ind].mean()
  
    
    df['AIS'].iloc[i]=  yais
   

    for j in range(1,19):
        y =-1*d3[f'{variable2}_sector_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = y[ind].mean()
        
    for j in range(1,4):
        y =-1*d3[f'{variable2}_region_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = y[ind].mean()
        
        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
df.to_csv(f'tables/{variable2}_2200.csv', index=False)


/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_61411/2435191546.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]=  yais
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_61411/2435191546.py:59: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sector_'+str(j)].iloc[i] = y[ind].mean()
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_61411/2435191546.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-vie

In [7]:
# avg and abs 2100
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    #get shelf area
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareafl/computed_iareafl_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    pth_tf = f'{pth_imip6hack}/{df.iloc[i].Experiment}/{variable2}/computed_{variable2}_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'

    #thermal foring 
    d3 = xr.open_dataset(pth_tf,decode_times=False )

    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )
    #
    ind = np.where((d.time.values >=2270) & (d.time.values <= 2300))
    
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =-d3[variable2].values[:tmin] #GT/m^3 yr

   
    yais =y[ind].mean()
  
    
    df['AIS'].iloc[i]=  yais
   

    for j in range(1,19):
        y =-1*d3[f'{variable2}_sector_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = y[ind].mean()
        
    for j in range(1,4):
        y =-1*d3[f'{variable2}_region_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = y[ind].mean()
        
        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
df.to_csv(f'tables/{variable2}_2300.csv', index=False)


/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_61411/4282634805.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]=  yais
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_61411/4282634805.py:59: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sector_'+str(j)].iloc[i] = y[ind].mean()
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_61411/4282634805.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-vie

averegeshelfmelt_2100.csv
averegeshelfmelt_2200.csv
averegeshelfmelt_2300.csv
best_ms_method_per_model.csv
best_ms_method_per_model_noAIS.csv
dslc_anom_2100.csv
dslc_anom_2200.csv
dslc_anom_2300.csv
iareafl_2100.csv
iareafl_2200.csv
iareafl_2300.csv
linear_meltsensetivity.csv
linear_meltsensetivity_TFBin.csv
linear_meltsensetivity_first50years.csv
linear_meltsensetivity_first50yearss.csv
linear_meltsensetivity_from_BMB_max.csv
linear_meltsensetivity_from_BMB_max_TFuntil2K.csv
linear_meltsensetivity_from_BMB_max_intercept.csv
linear_meltsensetivity_from_BMB_max_intercept_TFuntil2K.csv
linear_meltsensetivity_from_BMB_max_intercept_time2100.csv
linear_meltsensetivity_from_BMB_max_intercept_time_2100.csv
linear_meltsensetivity_from_BMB_max_time2100.csv
linear_meltsensetivity_from_ms_max.csv
mean_linear_meltsensetivity_from_BMB_max.csv
precentage_change_ofIceShelf.csv
precentage_change_ofIceShelf_2100.csv
precentage_change_ofIceShelf_2200.csv
precentage_change_ofIceShelf_2300.csv
prmse_reca